In [ ]:
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt

In [ ]:
# problem description

# We are given some forecasts for demands for our products.
# We want to optimise how much of each product to store at any given point in time 
# in order to maximise profit (maximum revenue and minimum cost).
# We only have limited amount of storage for each product category ()
# and each product carries a penalty cost for storing excess product in the warehouse.

In [ ]:
# Constants

INPUT_BASE_PATH = "../data"
MAX_TRAINING_TIMESTAMP = 1941
FORECASTS_PATH = Path("../results/ets/submissions")
FORECAST_HORIZON = 28
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

FOODS_PRODUCT_IDS = [
    "FOODS_3_555_TX_2_evaluation",
    "FOODS_3_376_TX_2_evaluation",
    "FOODS_3_811_CA_2_evaluation",
    "FOODS_1_218_TX_1_evaluation",
    "FOODS_3_226_WI_3_evaluation",
    "FOODS_3_070_WI_2_evaluation",
    "FOODS_3_007_TX_2_evaluation",
    "FOODS_3_444_TX_1_evaluation",
    "FOODS_2_398_WI_3_evaluation",
    "FOODS_3_540_WI_1_evaluation",
]
HOUSEHOLD_PRODUCT_IDS = [
    "HOUSEHOLD_1_334_TX_1_evaluation",
    "HOUSEHOLD_1_459_CA_2_evaluation",
    "HOUSEHOLD_2_342_WI_2_evaluation",
    "HOUSEHOLD_1_465_TX_3_evaluation",
    "HOUSEHOLD_1_294_WI_2_evaluation",
    "HOUSEHOLD_2_176_CA_3_evaluation",
    "HOUSEHOLD_1_334_TX_2_evaluation",
    "HOUSEHOLD_1_106_WI_2_evaluation",
    "HOUSEHOLD_1_474_TX_2_evaluation",
    "HOUSEHOLD_1_106_TX_1_evaluation",
]
HOBBIES_PRODUCT_IDS = [
    "HOBBIES_1_048_WI_1_evaluation",
    "HOBBIES_1_067_CA_3_evaluation",
    "HOBBIES_1_158_TX_3_evaluation",
    "HOBBIES_1_404_WI_3_evaluation",
    "HOBBIES_1_234_CA_3_evaluation",
    "HOBBIES_1_254_CA_3_evaluation",
    "HOBBIES_1_019_WI_1_evaluation",
    "HOBBIES_1_370_WI_1_evaluation",
    "HOBBIES_1_048_CA_1_evaluation",
    "HOBBIES_1_354_TX_3_evaluation",
]
ALL_PRODUCT_IDS = list(FOODS_PRODUCT_IDS + HOUSEHOLD_PRODUCT_IDS + HOBBIES_PRODUCT_IDS)

In [ ]:
# Load data and forecasts

SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
FORECASTS = pl.read_csv(f"{FORECASTS_PATH}/submission_2026-07-12 21:48:22.csv")

In [ ]:
# Prepare forecast df

forecast_df = (
    FORECASTS.filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .unpivot(index="id", value_name="sales", variable_name="F")
    .with_columns(
        item_store_id=pl.col("id").str.strip_suffix("_evaluation"),
        dept_id=pl.col("id").str.extract(r"^([^_]+)"),
        F_index=pl.col("F").str.strip_chars_start("F").cast(pl.Int64)
    )
    .with_columns(
        d=pl.format("d_{}", pl.col("F_index") + MAX_TRAINING_TIMESTAMP),
        d_index=pl.col("F_index") + MAX_TRAINING_TIMESTAMP
    )
    .select(["id", "item_store_id", "dept_id", "F", "F_index", "d", "d_index", "sales"])
)

forecast_df

In [ ]:
# Join calendar and price data onto forecasts

full_df = (
    forecast_df
    .join(
        CALENDAR_DATA.select(["d", "date", "wm_yr_wk"]),
        how="left",
        on="d"
    )
    .join(
        SELL_PRICES.with_columns(item_store_id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id"))),
        how="left",
        on=["item_store_id", "wm_yr_wk"]
    )
    .select(["id", "item_id", "store_id", "item_store_id", "dept_id", "F", "F_index", "d", "d_index", "date", "wm_yr_wk", "sales", "sell_price"])
)

full_df.head()

In [ ]:
# Define storage volume per product

# Define total storage volume

# Define total storage cost limit.

